# Soil Data Download — SoilGrids

Ask for soil in plain language — *"the soil a crop model needs for Yoro, Honduras"* — and get a
validated cube. A Gemini agent loads the
[`soil-data-download`](https://github.com/CGIAR-Climate-Data-Hub/skills) skill, resolves your
country or district to a bounding box, settles which properties and depths you need, and this
notebook downloads the SoilGrids rasters, stacks them into one cube, and plots the depth profiles.

---

### What SoilGrids gives you

Global rasters over five standard depths: `0-5`, `5-15`, `15-30`, `30-60`, `60-100` cm. The
default variable set is the one a process-based crop model needs — the seven general properties
plus the three water-retention points.

**The two groups come at different resolutions**, and that is not a detail you can ignore:

| Variable | What it is | Resolution | Stored as | Read as |
|---|---|---|---|---|
| `clay` / `sand` / `silt` | Texture fractions | 250 m | g/kg | ÷10 → % |
| `bdod` | Bulk density | 250 m | cg/cm³ | ÷100 → g/cm³ |
| `cfvo` | Coarse fragments | 250 m | cm³/dm³ | ÷10 → % |
| `soc` | Organic carbon | 250 m | dg/kg | ÷100 → % |
| `phh2o` | pH in water | 250 m | pH × 10 | ÷10 → pH |
| `nitrogen` | Total nitrogen | 250 m | cg/kg | ÷100 → g/kg |
| `cec` | Cation exchange capacity | 250 m | mmol(c)/kg | ÷10 → cmol(c)/kg |
| `wv0010` | Water at 10 kPa (near saturation) | **1 km** | 10⁻³ cm³/cm³ | ÷1000 → cm³/cm³ |
| `wv0033` | Water at 33 kPa — **field capacity** | **1 km** | 10⁻³ cm³/cm³ | ÷1000 → cm³/cm³ |
| `wv1500` | Water at 1500 kPa — **wilting point** | **1 km** | 10⁻³ cm³/cm³ | ÷1000 → cm³/cm³ |

The nine general properties are served by the 250 m coverage API. The three `wv*` water-retention
layers are **only published at 1 km** — there is no 250 m version of them to ask for.

Those water layers are why a crop model cares: **field capacity minus wilting point is the water a
plant can actually use**, and this notebook computes it for you as plant-available water over the
profile, in mm.

> **Mixing the two means resampling.** One cube has one grid, so the `reference_variable` you pick
> decides it. Name a 250 m variable and the 1 km water layers are **upsampled** onto it — you get
> 250 m pixels, but each 1 km value is simply repeated across 16 of them; the detail is replicated,
> not measured. Name `wv0033` instead and the 250 m layers are averaged down to 1 km — nothing is
> invented, but texture detail is lost. Neither is wrong; it depends on whether your model wants a
> fine grid or honest cell values. This notebook tells you which way it went.

`ocd` (organic carbon density) and `ocs` (organic carbon stocks) appear in the SoilGrids catalogue
but are **not wired into aggeodata** — asking for them logs a warning and writes nothing, so the
cube would come back quietly missing that layer. They are left out here rather than offered.

> **The CRS is the thing that bites.** SoilGrids ships in Interrupted Goode Homolosine, in metres.
> A cube left in that projection pairs with nothing: coordinates read as 1670931 rather than
> 15.06, and a crop model silently skips every pixel. This notebook reprojects to `EPSG:4326`
> explicitly and then checks the result is actually in degrees.

### How to use this notebook

1. **Runtime ▸ Run all.** Setup takes about two minutes.
2. Edit the request in section **4** and answer the agent's questions.
3. The maps and profiles appear in section **5**.

### Before you start

| Requirement | How |
|---|---|
| Gemini API key | Create one at [aistudio.google.com](https://aistudio.google.com/apikey), then add it in Colab under **Secrets** as `GOOGLE_API_KEY` with notebook access enabled. |
| Nothing else | SoilGrids is open — no account, no key, no rate limit to work around. |

> **Size check before you ask for a country.** The general properties are 250 m, so a whole country
> is a lot of tiles: ten variables over five depths is 50 rasters, and the extent decides how big
> each one is.
> A district downloads in a minute or two; a large country can take much longer. The agent reports
> the pixel count before starting so you can decide.


---
## 1 · Setup


In [1]:
#@title Install aggeodata and the Gemini SDK { display-mode: "form" }
%pip install -q "aggeodata[download] @ git+https://github.com/CGIAR-Climate-Data-Hub/aggeodata.git"
%pip install -q -U "google-genai>=2.9.0"
print("Install finished.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 58.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.7/221.7 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.3/405.3 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.6/102.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 42.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.3/241.3 kB 

In [2]:
#@title Environment check { display-mode: "form" }
import importlib.metadata as md_, pathlib

# Paths FIRST, before anything that can stop this cell. Defining them after a guard
# means a failed check leaves them unbound, and the symptom is a NameError much
# later in the run rather than the message printed here.
OUT_DIR = pathlib.Path("/content/soil_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

missing = []
for pkg in ("aggeodata", "rioxarray", "xarray"):
    try:
        print(f"OK       {pkg} {md_.version(pkg)}")
    except md_.PackageNotFoundError:
        print(f"MISSING  {pkg}")
        missing.append(pkg)
if missing:
    raise RuntimeError(f"Missing: {', '.join(missing)}. Re-run the install cell; if it "
                       "still fails, use Runtime > Restart session and run all.")

print(f"\noutput: {OUT_DIR}")

OK       aggeodata 0.2.3
OK       rioxarray 0.23.0
OK       xarray 2026.9.0

output: /content/soil_data


In [3]:
#@title Connect to the Gemini API { display-mode: "form" }
SECRET_NAME = "GOOGLE_API_KEY2" #@param {type:"string"}

import uuid, json
from google import genai
from google.colab import userdata
from IPython.display import display, Markdown
import ipywidgets as widgets

client = genai.Client(api_key=userdata.get(SECRET_NAME))
AGENT  = "antigravity-preview-05-2026"
print("Connected.")

TimeoutException: Requesting secret GOOGLE_API_KEY2 timed out. Secrets can only be fetched when running from the Colab UI.

---
## 2 · The pipeline

Three steps, matching the skill: resolve the area, download the rasters, stack and validate.

The build step passes the CRS pair explicitly — `+proj=igh` in, `EPSG:4326` out. Recent aggeodata
defaults these correctly, but a cube that keeps the projected CRS fails downstream in ways that
look like missing data rather than a coordinate problem, so it is worth being loud about.

The cube is validated before you ever see it: coordinates in degrees, every requested variable
present, texture fractions summing to ~1000 g/kg, and the water-retention points ordered
`wv1500 < wv0033 < wv0010`. A cube that fails those is reported rather than returned quietly.


In [ ]:
#@title Pipeline - resolve extent, download, build, validate { display-mode: "form" }
import time, traceback
import numpy as np
import pandas as pd
import xarray as xr

# The crop-modeling set: seven general properties plus the three water-retention
# points. wv0033 - wv1500 is the water a plant can actually use.
CROP_VARS  = ["clay", "sand", "silt", "bdod", "cfvo", "soc", "phh2o",
              "wv0010", "wv0033", "wv1500"]
# ocd and ocs are routed by NEITHER endpoint: the downloader logs "Unknown
# SoilGrids variable" and writes nothing, so a run "succeeds" with them missing.
# They are excluded rather than offered.
ALL_VARS   = CROP_VARS + ["nitrogen", "cec"]
UNAVAILABLE = {"ocd": "organic carbon density", "ocs": "organic carbon stocks"}
# The 250 m coverage API vs the 1 km aggregated hydraulic layers.
RES_M      = {v: 250 for v in ["clay", "sand", "silt", "bdod", "cfvo", "soc",
                               "phh2o", "nitrogen", "cec"]}
RES_M.update({v: 1000 for v in ["wv0010", "wv0033", "wv1500"]})
DEPTHS     = ["0-5", "5-15", "15-30", "30-60", "60-100"]
# cm of soil each layer represents, for depth-weighted totals
THICKNESS  = {"0-5": 5, "5-15": 10, "15-30": 15, "30-60": 30, "60-100": 40}

# SoilGrids stores integers. These are the divisors that turn them into the units
# a person reads, and are the same ones ag-cube-cm applies when writing DSSAT files.
TO_READABLE = {
    "clay": (10, "%"), "sand": (10, "%"), "silt": (10, "%"), "cfvo": (10, "%"),
    "bdod": (100, "g/cm3"), "soc": (100, "%"), "phh2o": (10, "pH"),
    "wv0010": (1000, "cm3/cm3"), "wv0033": (1000, "cm3/cm3"), "wv1500": (1000, "cm3/cm3"),
    "nitrogen": (100, "g/kg"), "cec": (10, "cmol(c)/kg"),
}

IGH = "+proj=igh +lat_0=0 +lon_0=0 +datum=WGS84 +units=m +no_defs"   # SoilGrids native

state = {}


def _say(msg):
    try:
        with out:
            print(msg)
    except NameError:
        print(msg)


def open_cube_any(path):
    """Open either format. A Zarr store is not a NetCDF file, and the plain zarr
    reader leaves spatial_ref as a data variable so .rio.crs returns None."""
    path = pathlib.Path(path)
    if path.suffix != ".zarr":
        return xr.open_dataset(path)
    try:
        return xr.open_zarr(str(path), consolidated=True, decode_coords="all")
    except (KeyError, ValueError, FileNotFoundError):
        return xr.open_zarr(str(path), consolidated=False, decode_coords="all")


def _axis(ds, *names):
    return next((d for d in names if d in ds.dims), None)


def list_admin_units(country_code, adm_level=1, **_):
    """The spellings the boundary file actually uses, so resolve_extent is not a guess."""
    try:
        from aggeodata.ingestion.boundaries import list_admin_units as _list
        names = _list(country_code, adm_level)
        return {"state": "ok", "count": len(names), "adm_level": adm_level,
                "units": names[:200], "truncated": len(names) > 200}
    except Exception as exc:
        return {"state": "error", "error": f"{type(exc).__name__}: {exc}"}


def resolve_extent(country_code=None, feature_name=None, adm_level=1, bbox=None, **_):
    """Country or admin unit -> [xmin, ymin, xmax, ymax], and how big that is at 250 m.

    250 m is the finest the download can be, so it is the honest worst case for sizing
    even when the cube ends up on the 1 km water grid.
    """
    try:
        if bbox:
            ext = [float(v) for v in bbox]
            src = "given"
        elif not country_code:
            return {"state": "need_input", "note": "give a country_code (ISO3) or a bbox"}
        else:
            from aggeodata.ingestion.boundaries import get_admin_boundary, _fetch_geojson_cached
            if feature_name:
                gdf = get_admin_boundary(country_code, feature_name, adm_level)
            else:
                try:
                    gdf = _fetch_geojson_cached(country_code, 0)
                except Exception:
                    gdf = _fetch_geojson_cached(country_code, 1)
            ext = [round(float(v), 4) for v in gdf.total_bounds.tolist()]
            src = f"{country_code} {feature_name or '(whole country)'} adm{adm_level}"

        # SoilGrids is 250 m ~ 0.00225 deg. The pixel count is the honest measure of
        # how long this will take, and it grows with the square of the extent.
        deg = 0.00225
        nx = max(int((ext[2] - ext[0]) / deg), 1)
        ny = max(int((ext[3] - ext[1]) / deg), 1)
        px = nx * ny
        return {"state": "ok", "bbox": ext, "source": src,
                "approx_pixels_per_raster": px,
                "size_note": ("small - a minute or two" if px < 4e6 else
                              "large - expect many minutes per variable; consider a "
                              "district instead of the whole country" if px < 6e7 else
                              "very large - strongly consider narrowing the area first")}
    except Exception as exc:
        return {"state": "error", "error": f"{type(exc).__name__}: {exc}",
                "note": "If the unit name is wrong, call list_admin_units for the spellings."}


def download_soil(bbox, suffix="soil", variables=None, depths=None,
                  output_format="zarr", reference_variable="wv1500", **_):
    """Download SoilGrids for a bbox, stack into one cube, and validate it."""
    try:
        return _download_soil(bbox, suffix, variables or CROP_VARS, depths or DEPTHS,
                              output_format, reference_variable)
    except Exception as exc:
        _say(f"Soil error: {exc}")
        return {"state": "error", "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc()[-1500:]}


def _download_soil(bbox, suffix, variables, depths, output_format, reference_variable):
    t0 = time.time()
    run_dir = OUT_DIR / suffix
    tiles = run_dir / "soil_raw"
    run_dir.mkdir(parents=True, exist_ok=True)

    blocked = [v for v in variables if v in UNAVAILABLE]
    if blocked:
        return {"state": "bad_request",
                "note": (f"{', '.join(blocked)} is not wired up in aggeodata - the downloader "
                         "skips it silently and the cube would simply be missing that layer. "
                         f"Available: {ALL_VARS}")}
    unknown = [v for v in variables if v not in ALL_VARS]
    if unknown:
        return {"state": "bad_request",
                "note": f"not SoilGrids variables: {unknown}. Available: {ALL_VARS}"}
    bad_depths = [d for d in depths if d not in DEPTHS]
    if bad_depths:
        return {"state": "bad_request",
                "note": f"not SoilGrids depths: {bad_depths}. Available: {DEPTHS}"}

    existing = sorted(tiles.glob("*.tif"))
    if existing:
        _say(f"{len(existing)} tile(s) already present - skipping the download")
    else:
        _say(f"downloading {len(variables)} variable(s) x {len(depths)} depth(s) "
             f"= {len(variables) * len(depths)} rasters ...")
        from aggeodata.ingestion.soil import SoilGridsDownloader
        got = SoilGridsDownloader(soil_layers=list(variables), depths=list(depths),
                                  output_folder=str(tiles)).download(boundaries=list(bbox))
        _say(f"  {len(got)} GeoTIFF tile(s)")

    _say("building the cube (+proj=igh -> EPSG:4326) ...")
    from aggeodata.transform.soil_cube import SoilDataCubeBuilder
    ext = ".zarr" if output_format == "zarr" else ".nc"
    out = pathlib.Path(SoilDataCubeBuilder(
        data_folder=str(tiles), variables=list(variables),
        reference_variable=reference_variable, crs=IGH, target_crs="EPSG:4326",
    ).build_and_save(output_path=str(run_dir), filename=f"soil_{suffix}{ext}",
                     output_format=output_format, codec="zstd"))

    ds = open_cube_any(out)
    # Older builds emitted one variable per depth (clay_0-5cm_mean) instead of a
    # depth axis. Reshape, because everything downstream indexes by depth.
    if [v for v in ds.data_vars if "_cm_mean" in v]:
        from aggeodata.transform.soil_cube import reshape_flat_soil_cube
        ds.close()
        out = out.with_suffix(".nc")
        reshape_flat_soil_cube(open_cube_any(out)).to_netcdf(out)
        ds = open_cube_any(out)
        _say("  reshaped a flat-format cube to a depth axis")

    report = validate_cube(ds, variables)
    res = sorted({RES_M.get(v) for v in variables if v in RES_M})
    mix = None
    if len(res) > 1:
        ref_res = RES_M.get(reference_variable, "?")
        mix = (f"This cube mixes {res[0]} m and {res[1]} m source layers. "
               f"reference_variable={reference_variable} ({ref_res} m) sets the grid, so the "
               + ("1 km water layers were upsampled - the extra pixels are replicated, not "
                  "measured." if ref_res == 250 else
                  "250 m layers were averaged down to 1 km - nothing invented, but texture "
                  "detail is lost."))
    summary = {"path": str(out), "format": "zarr" if out.suffix == ".zarr" else "netcdf",
               "variables": [v for v in ds.data_vars if v != "spatial_ref"],
               "dims": {k: int(v) for k, v in ds.sizes.items()},
               "depths": [str(v) for v in ds.depth.values] if "depth" in ds.dims else [],
               "bbox": report.pop("bbox", None)}
    ds.close()
    state.update(cube=out, suffix=suffix, variables=list(variables))
    _say(f"  -> {out.name}  {summary['dims']}")
    _say(f"validation: {report['verdict']}")
    if mix:
        _say(f"note: {mix}")
    return {"state": "ok" if report["verdict"] == "PASSED" else "problems",
            "cube": summary, "validation": report, "resolution_note": mix,
            "source_resolution_m": {v: RES_M.get(v) for v in variables if v in RES_M},
            "minutes": round((time.time() - t0) / 60, 1)}


def validate_cube(ds, expected):
    """The skill's Step 5 checks, plus two the soil data itself makes possible."""
    problems, checks = [], {}
    ydim, xdim = _axis(ds, "y", "lat", "latitude"), _axis(ds, "x", "lon", "longitude")

    # 1. degrees, not projected metres - the SoilGrids CRS trap
    if ydim and xdim:
        in_deg = abs(float(ds[ydim].min())) < 90 and abs(float(ds[xdim].min())) < 180
        checks["coords_in_degrees"] = in_deg
        if not in_deg:
            problems.append("coordinates are in metres - the cube kept the IGH projection, "
                            "so it will not pair with any climate cube")
        else:
            checks["bbox"] = [round(float(ds[xdim].min()), 4), round(float(ds[ydim].min()), 4),
                              round(float(ds[xdim].max()), 4), round(float(ds[ydim].max()), 4)]

    # 2. every requested variable arrived
    missing = [v for v in expected if v not in ds.data_vars]
    checks["all_variables_present"] = not missing
    if missing:
        problems.append(f"missing variables: {missing}")

    # 3. texture fractions should sum to ~1000 g/kg; a big miss means a bad tile
    if all(v in ds.data_vars for v in ("clay", "sand", "silt")):
        total = float(np.nanmean((ds.clay + ds.sand + ds.silt).values))
        checks["texture_sums_to_1000_g_per_kg"] = round(total, 1)
        if not 900 <= total <= 1100:
            problems.append(f"clay+sand+silt averages {total:.0f} g/kg, not ~1000 - "
                            "suspect a corrupt or mismatched tile")

    # 4. the water-retention points must be physically ordered
    if all(v in ds.data_vars for v in ("wv1500", "wv0033", "wv0010")):
        ok = bool(np.nanmean(ds.wv1500.values) < np.nanmean(ds.wv0033.values)
                  < np.nanmean(ds.wv0010.values))
        checks["wilting < field capacity < saturation"] = ok
        if not ok:
            problems.append("water-retention points are out of order - a crop model would "
                            "compute negative available water from this")

    return {"verdict": "PASSED" if not problems else "FAILED",
            "checks": checks, "problems": problems}


def available_water(**_):
    """Plant-available water over the profile, in mm.

    (field capacity - wilting point) x layer thickness, summed over depths. This is
    the number that decides how long a crop can go between rains, and it is the
    reason a crop model wants wv0033 and wv1500 rather than texture alone.
    """
    if not state.get("cube"):
        return {"state": "nothing_yet", "note": "download a cube first"}
    ds = open_cube_any(state["cube"])
    if not {"wv0033", "wv1500"} <= set(ds.data_vars):
        ds.close()
        return {"state": "unavailable",
                "note": "needs wv0033 and wv1500; re-download including both"}
    per_depth, total = {}, None
    for d in [str(v) for v in ds.depth.values]:
        awc = (ds.wv0033.sel(depth=d) - ds.wv1500.sel(depth=d)) / 1000.0   # cm3/cm3
        mm = awc * THICKNESS.get(d, 0) * 10                                 # cm -> mm
        per_depth[d] = round(float(np.nanmean(mm.values)), 1)
        total = mm if total is None else total + mm
    state["awc_mm"] = total
    result = {"state": "ok", "per_depth_mm": per_depth,
              "profile_total_mm": round(float(np.nanmean(total.values)), 1),
              "min_mm": round(float(np.nanmin(total.values)), 1),
              "max_mm": round(float(np.nanmax(total.values)), 1),
              "depths_cm": {d: THICKNESS.get(d) for d in per_depth}}
    ds.close()
    return result


print("Pipeline loaded.")

In [ ]:
#@title Plotting - depth profiles, maps per depth, available water { display-mode: "form" }
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 10, "font.size": 9})

CMAP = {"clay": "Oranges", "sand": "YlOrBr", "silt": "BuPu", "soc": "Greens",
        "phh2o": "RdYlBu", "bdod": "copper", "cfvo": "Greys",
        "wv0010": "Blues", "wv0033": "Blues", "wv1500": "Blues"}


def _panels(n):
    cols = min(n, 5)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.1 * cols, 2.9 * rows), squeeze=False)
    return fig, [a for row in axes for a in row]


def _readable(ds, var):
    """Divide out the SoilGrids integer scaling so axes carry real units."""
    div, unit = TO_READABLE.get(var, (1, ""))
    return ds[var] / div, unit


def plot_soil(variables=None, **_):
    """One row of maps per variable across depths, plus the profile means."""
    if not state.get("cube"):
        return {"state": "nothing_to_plot", "note": "no cube yet"}
    ds = open_cube_any(state["cube"])
    depths = [str(v) for v in ds.depth.values] if "depth" in ds.dims else []
    wanted = [v for v in (variables or state.get("variables") or list(ds.data_vars))
              if v in ds.data_vars and v != "spatial_ref"]

    summary = {}
    for var in wanted:
        da, unit = _readable(ds, var)
        fig, axes = _panels(len(depths))
        vmin = float(np.nanpercentile(da.values, 2))
        vmax = float(np.nanpercentile(da.values, 98))
        im = None
        for ax, d in zip(axes, depths):
            im = da.sel(depth=d).plot(ax=ax, cmap=CMAP.get(var, "viridis"),
                                      vmin=vmin, vmax=vmax, add_colorbar=False)
            ax.set_title(f"{d} cm"); ax.set_xlabel(""); ax.set_ylabel("")
        for ax in axes[len(depths):]:
            ax.axis("off")
        if im is not None:
            fig.colorbar(im, ax=axes, shrink=0.7, label=f"{var} ({unit})")
        fig.suptitle(f"{var} by depth - {state['suffix']}", y=1.02)
        png = pathlib.Path(state["cube"]).parent / f"{var}_by_depth_{state['suffix']}.png"
        fig.savefig(png, bbox_inches="tight"); plt.show()
        summary[var] = {"unit": unit,
                        "by_depth": {d: round(float(np.nanmean(da.sel(depth=d).values)), 2)
                                     for d in depths},
                        "png": str(png)}

    # The profile view: every variable against depth, which is how a soil scientist
    # reads a pit face - shallow at the top.
    if depths:
        fig, axes = plt.subplots(1, len(wanted), figsize=(2.1 * len(wanted), 3.6), squeeze=False)
        mid = [sum(map(int, d.split("-"))) / 2 for d in depths]
        for ax, var in zip(axes[0], wanted):
            da, unit = _readable(ds, var)
            ax.plot([float(np.nanmean(da.sel(depth=d).values)) for d in depths], mid, "o-")
            ax.invert_yaxis()
            ax.set_title(f"{var}\n({unit})", fontsize=8)
            ax.set_xlabel(""); ax.grid(alpha=.3)
        axes[0][0].set_ylabel("depth (cm)")
        fig.suptitle(f"Mean profile - {state['suffix']}", y=1.04)
        png = pathlib.Path(state["cube"]).parent / f"profile_{state['suffix']}.png"
        fig.savefig(png, bbox_inches="tight"); plt.show()
        summary["_profile_png"] = str(png)
    ds.close()

    # Plant-available water, the crop-modeling payoff
    awc = available_water()
    if awc.get("state") == "ok" and state.get("awc_mm") is not None:
        fig, ax = plt.subplots(figsize=(4.4, 3.6))
        state["awc_mm"].plot(ax=ax, cmap="YlGnBu",
                             cbar_kwargs={"label": "plant-available water (mm)"})
        ax.set_title(f"Available water, 0-100 cm - {state['suffix']}")
        ax.set_xlabel(""); ax.set_ylabel("")
        png = pathlib.Path(state["cube"]).parent / f"available_water_{state['suffix']}.png"
        fig.savefig(png, bbox_inches="tight"); plt.show()
        awc["png"] = str(png)
    return {"state": "ok", "variables": summary, "available_water": awc}


print("Plotting loaded.")

---
## 3 · Agent tools

Five functions run **here in the notebook**, not in the agent's sandbox — the sandbox has no
keyboard, no open internet and nowhere to put the rasters:

- **`ask_user`** — a batch of questions, answered one at a time, returned together.
- **`list_admin_units`** / **`resolve_extent`** — the spellings a country uses, and the bounding
  box with an honest pixel count.
- **`download_soil`** — the rasters, the cube, and the validation.
- **`available_water`** — field capacity minus wilting point over the profile, in mm.
- **`plot_soil`** — maps per depth, the mean profile, and the available-water map.


In [ ]:
#@title Tool declarations { display-mode: "form" }
ask_user_tool = {
    "type": "function",
    "name": "ask_user",
    "description": (
        "Ask the user for a BATCH of parameters in a single call and receive every answer "
        "together. Put EVERY field you still need into `questions` - the client presents them "
        "one at a time and returns all the answers in one response, so a batch of 10 costs "
        "exactly what a batch of 1 costs. Calling this with one question and waiting burns the "
        "user's quota and is the most common way this workflow fails."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "questions": {
                "type": "array",
                "description": "Every parameter you need right now. Longer is better.",
                "items": {
                    "type": "object",
                    "properties": {
                        "field":      {"type": "string", "description": "Parameter, e.g. 'depths'."},
                        "question":   {"type": "string", "description": "The question, in plain language."},
                        "suggestion": {"type": "string", "description": "Draft value the user accepts with Enter."},
                    },
                    "required": ["field", "question"],
                },
            },
        },
        "required": ["questions"],
    },
}

list_admin_units_tool = {
    "type": "function",
    "name": "list_admin_units",
    "description": (
        "List the administrative unit names a country actually uses, at a given level "
        "(1 = region/province, 2 = district). Call this BEFORE resolve_extent whenever the user "
        "names a sub-national area, so you pass a spelling the boundary file contains rather "
        "than guessing."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "country_code": {"type": "string", "description": "ISO 3166-1 alpha-3, e.g. HND."},
            "adm_level":    {"type": "integer", "description": "1 = region/province, 2 = district."},
        },
        "required": ["country_code"],
    },
}

resolve_extent_tool = {
    "type": "function",
    "name": "resolve_extent",
    "description": (
        "Turn a country, or one admin unit inside it, into [xmin, ymin, xmax, ymax]. Pass a bbox "
        "straight through if the user gave one. Returns `approx_pixels_per_raster` and a "
        "`size_note`: the general properties are 250 m, so area drives the download time more than the number "
        "of variables does. Show the user the bbox and the size note before downloading, and if "
        "the note says the area is large, say so and offer a district instead."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "country_code": {"type": "string", "description": "ISO 3166-1 alpha-3, e.g. HND."},
            "feature_name": {"type": "string", "description": "One admin unit, spelled as list_admin_units returned it."},
            "adm_level":    {"type": "integer", "description": "Level the feature_name belongs to."},
            "bbox":         {"type": "array", "items": {"type": "number"},
                             "description": "[xmin, ymin, xmax, ymax] if the user gave one."},
        },
        "required": [],
    },
}

download_soil_tool = {
    "type": "function",
    "name": "download_soil",
    "description": (
        "Download SoilGrids for a bounding box, stack it into one cube, and validate it. Runs in "
        "the notebook. SoilGrids is static, so there is no date range - just the area, the "
        "variables and the depths. Call ONCE, after the user confirms the plan.\n"
        "The defaults are the crop-modeling set: clay, sand, silt, bdod, cfvo, soc, phh2o, "
        "wv0010, wv0033, wv1500 over all five depths. Narrow them only if the user asks - the "
        "three wv* water-retention variables in particular are what a crop model needs, so do "
        "not drop them from a crop-modeling request.\n"
        "Returns a validation report. If the verdict is FAILED, relay the problems and stop "
        "rather than plotting."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "bbox":      {"type": "array", "items": {"type": "number"},
                          "description": "[xmin, ymin, xmax, ymax] from resolve_extent."},
            "suffix":    {"type": "string", "description": "Short label, no spaces, e.g. 'yoro_hnd'."},
            "variables": {"type": "array", "items": {"type": "string"},
                          "description": "SoilGrids names: clay, sand, silt, bdod, cfvo, soc, phh2o, nitrogen, cec (250 m); wv0010, wv0033, wv1500 (1 km). ocd and ocs are not available."},
            "depths":    {"type": "array", "items": {"type": "string"},
                          "description": "Any of '0-5', '5-15', '15-30', '30-60', '60-100' (cm)."},
            "output_format": {"type": "string", "description": "'zarr' (default) or 'netcdf'."},
            "reference_variable": {"type": "string", "description": "Variable whose grid defines the cube; the others are resampled onto it. Default wv1500 (1 km), which avoids upsampling the water layers."},
        },
        "required": ["bbox", "suffix"],
    },
}

available_water_tool = {
    "type": "function",
    "name": "available_water",
    "description": (
        "Plant-available water over the soil profile, in mm: (field capacity - wilting point) "
        "times each layer's thickness, summed. This is the number that decides how long a crop "
        "can go between rains, so quote it when the user's purpose is crop modelling. Needs "
        "wv0033 and wv1500 in the cube."
    ),
    "parameters": {"type": "object", "properties": {}, "required": []},
}

plot_soil_tool = {
    "type": "function",
    "name": "plot_soil",
    "description": (
        "Draw the maps: one row per variable across the depths, the mean depth profile, and the "
        "available-water map. Call once, after a cube validates. Returns the numbers behind "
        "every panel so you can describe them - you cannot see the images."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "variables": {"type": "array", "items": {"type": "string"},
                          "description": "Omit to plot everything in the cube."},
        },
        "required": [],
    },
}

TOOLS = [ask_user_tool, list_admin_units_tool, resolve_extent_tool,
         download_soil_tool, available_water_tool, plot_soil_tool]
print("Tools declared:", [t["name"] for t in TOOLS])

In [ ]:
#@title Chat plumbing - the question box and the turn loop { display-mode: "form" }
import re

MAX_RETRIES = 4

out  = widgets.Output()
box  = widgets.Text(layout=widgets.Layout(width="80%"))
send = widgets.Button(description="Send", button_style="primary")
ui   = widgets.HBox([box, send])

state.update({"result": None, "env": None, "queue": [], "answers": [],
              "subq": None, "collected": {}, "batch": 0, "turns": 0})

LOCAL = {"list_admin_units": list_admin_units, "resolve_extent": resolve_extent,
         "download_soil": download_soil, "available_water": available_water,
         "plot_soil": plot_soil}


def api_call(**kwargs):
    """interactions.create with backoff, so a free-tier 429 pauses instead of crashing."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            res = client.interactions.create(**kwargs)
            state["turns"] += 1
            return res
        except Exception as exc:
            text = str(exc)
            if "429" not in text and "too_many_requests" not in text.lower():
                raise
            found = re.search(r"retry in ([\d.]+)s", text)
            wait  = float(found.group(1)) + 2 if found else 20 * attempt
            with out:
                print(f"rate limited - waiting {wait:.0f}s "
                      f"(attempt {attempt}/{MAX_RETRIES}; the notebook will resume on its own)")
            time.sleep(wait)
    raise RuntimeError("Still rate limited after several retries - wait a minute and re-run, "
                       "or use a paid key.")


def show(obj, prefix="**Agent**\n\n"):
    text = obj if isinstance(obj, str) else getattr(obj, "output_text", None)
    if text:
        display(Markdown(prefix + text))


def pending_calls(res):
    done = {s.call_id for s in res.steps if s.type == "function_result"}
    return [s for s in res.steps if s.type == "function_call" and s.id not in done]


def call_args(call):
    return call.arguments if isinstance(call.arguments, dict) else json.loads(call.arguments)


def reply(call, payload):
    state["answers"].append({"type": "function_result", "name": call.name,
                             "call_id": call.id, "result": payload})


def end(msg):
    box.disabled = send.disabled = True
    with out:
        display(Markdown(f"### {msg}"))


def run_local(call):
    """Execute a non-interactive tool call here. Nothing is allowed to raise: a tool
    that dies would unwind the loop and leave the chat box disabled for good."""
    fn = LOCAL.get(call.name)
    if fn is None:
        return {"error": f"no local handler for {call.name}"}
    with out:
        display(Markdown(f"*running `{call.name}`...*"))
    try:
        res = fn(**call_args(call))
    except Exception as exc:
        tb = traceback.format_exc()
        with out:
            display(Markdown(f"**`{call.name}` failed** — {type(exc).__name__}: {exc}"))
            print(tb[-1500:])
        return {"state": "error", "tool": call.name,
                "error": f"{type(exc).__name__}: {exc}", "traceback": tb[-1500:]}
    if not isinstance(res, dict):
        res = {"state": "ok", "result": res}
    with out:
        if call.name == "resolve_extent" and res.get("bbox"):
            display(Markdown(f"- bbox `{res['bbox']}` — {res.get('source','')} "
                             f"(~{res.get('approx_pixels_per_raster','?'):,} px per raster, "
                             f"{res.get('size_note','')})"))
        elif call.name == "list_admin_units" and res.get("units"):
            display(Markdown(f"- {res['count']} units at adm{res['adm_level']}: "
                             + ", ".join(res["units"][:12])
                             + (" …" if res["count"] > 12 else "")))
        elif call.name == "download_soil" and res.get("validation"):
            v = res["validation"]
            display(Markdown(f"**Validation {v['verdict']}** — "
                             + ("; ".join(v["problems"]) if v["problems"] else "all checks passed")))
        elif call.name == "available_water" and res.get("profile_total_mm") is not None:
            display(Markdown(f"- plant-available water 0–100 cm: "
                             f"**{res['profile_total_mm']} mm** "
                             f"(range {res['min_mm']}–{res['max_mm']})"))
    return res


def questions_of(call):
    args  = call_args(call)
    items = args.get("questions")
    if isinstance(items, list) and items:
        return [q for q in items if isinstance(q, dict) and q.get("question")]
    if args.get("question"):
        return [{"field": args.get("field", "value"), "question": args["question"],
                 "suggestion": args.get("suggestion")}]
    return []


def pose(q):
    done    = state["batch"] - len(state["subq"]) + 1
    counter = f"`{done}/{state['batch']}` " if state["batch"] > 1 else ""
    text    = f"{counter}**{q.get('field', 'question')}** - {q['question']}"
    if q.get("suggestion"):
        text += (f"\n\n> *Proposed:* {q['suggestion']}\n>\n"
                 "> *Press Enter on an empty box to accept, or type a replacement.*")
    with out:
        display(Markdown(text))
    box.placeholder = "Enter to accept" if q.get("suggestion") else "Your answer..."
    box.disabled = send.disabled = False
    box.value = ""


def advance_turn():
    box.disabled = send.disabled = True
    with out:
        print(f"thinking... (turn {state['turns'] + 1}, sending {len(state['answers'])} result(s))")
    state["result"] = api_call(agent=soil_agent.id,
                               previous_interaction_id=state["result"].id,
                               environment=state["env"], input=state["answers"], tools=TOOLS)
    state["answers"] = []
    with out:
        show(state["result"])
    ask_next()


def ask_next():
    """Drive the queue: run local tools silently, pause only for questions."""
    while True:
        while state["queue"] and state["queue"][0].name != "ask_user":
            call = state["queue"].pop(0)
            reply(call, run_local(call))

        if state["queue"]:
            call = state["queue"][0]
            if state["subq"] is None:
                state["subq"]      = questions_of(call)
                state["collected"] = {}
                state["batch"]     = len(state["subq"])
                if state["batch"] > 1:
                    with out:
                        display(Markdown(f"*{state['batch']} questions in this batch - answering "
                                         "all of them costs a single API turn.*"))
            if state["subq"]:
                return pose(state["subq"][0])
            state["queue"].pop(0)
            reply(call, {"answers": state["collected"]})
            state["subq"] = None
            continue

        if state["answers"]:
            return advance_turn()

        res = state["result"]
        if res.status != "requires_action":
            return end(f"Finished in {state['turns']} API turns.")
        calls = pending_calls(res)
        if not calls:
            return end(f"Finished in {state['turns']} API turns - nothing left to answer.")
        state["queue"] = calls


def submit(_):
    try:
        _submit()
    except Exception as exc:
        with out:
            display(Markdown(f"**The run stopped** — {type(exc).__name__}: {exc}"))
            print(traceback.format_exc()[-1500:])
            display(Markdown("Re-run the `start(PROMPT)` cell to try again. "
                             "Downloaded tiles and any finished cube are still on disk."))
        box.disabled = send.disabled = False


def _submit():
    if not state["subq"] or box.disabled:
        return
    msg, box.value = box.value.strip(), ""
    q = state["subq"].pop(0)
    answer = msg or q.get("suggestion") or ""
    with out:
        display(Markdown(f"**You** - {answer}"))
    state["collected"][q.get("field", f"q{len(state['collected'])}")] = answer
    ask_next()


send.on_click(submit)
box.on_submit(submit)   # Enter key (deprecated in ipywidgets 8 - harmless warning)


def start(first_input):
    try:
        _start(first_input)
    except Exception as exc:
        display(Markdown(f"**Could not start** — {type(exc).__name__}: {exc}"))
        print(traceback.format_exc()[-2000:])


def _start(first_input):
    out.clear_output()
    state.update(result=None, env=None, queue=[], answers=[], subq=None,
                 collected={}, batch=0, turns=0)
    state["result"] = api_call(agent=soil_agent.id, input=first_input,
                               environment="remote", tools=TOOLS)
    state["env"] = state["result"].environment_id
    display(out, ui)
    with out:
        show(state["result"])
    ask_next()


print("Helpers loaded.")

---
## 4 · The agent &nbsp; ✏️ **EDIT ME**

The standing brief, with the full `SKILL.md` appended so the agent never spends a turn reading a
file. Re-run the agent cell after editing it — the brief is baked in at creation time.


In [ ]:
#@title Build the standing brief { display-mode: "form" }
import urllib.request

# Bound before the fetch so a network failure cannot leave them undefined: the
# agent cell would then raise NameError instead of saying what actually went wrong.
SYSTEM_INSTRUCTION = None
SKILL_TEXT = ""

_SKILL_URL = ("https://raw.githubusercontent.com/CGIAR-Climate-Data-Hub/skills/main/"
              ".agents/skills/soil-data-download/SKILL.md")
try:
    with urllib.request.urlopen(urllib.request.Request(
            _SKILL_URL, headers={"User-Agent": "cdh-soil-notebook/1.0"}), timeout=30) as _r:
        SKILL_TEXT = _r.read().decode("utf-8")
except Exception as _exc:
    print(f"Could not fetch the skill from {_SKILL_URL}")
    print(f"  {type(_exc).__name__}: {_exc}")
    print("  Check connectivity and re-run this cell - the agent needs it.")

BRIEF = """
You are a soil data assistant for the CGIAR Climate Data Hub. You download SoilGrids for an area
and explain what the soil is like. The full soil-data-download skill is appended below. Follow its
workflow, with five differences that apply in this notebook.

1. THE NOTEBOOK DOWNLOADS, NOT YOU. Your sandbox has no internet and nowhere to put rasters.
   `download_soil` runs the downloader, builds the cube with the CRS pair the skill insists on,
   and validates the result. Never write configs or shell commands, and never ask the user to run
   anything.

2. AREA FIRST, AND SIZE IS THE DECISION. The general soil properties are 250 m, so the extent
   drives the download time far more than the variable count. For a sub-national request call `list_admin_units`
   first, then `resolve_extent` with a spelling that list returned. Always show the bbox AND the
   size note before downloading; if the area is large, say what that means in minutes and offer a
   district as an alternative rather than starting a long download silently.

3. THE DEFAULTS ARE THE CROP-MODELING SET. Ten variables over five depths. Keep them unless the
   user asks to narrow. In particular wv0010, wv0033 and wv1500 are the water-retention points -
   if the user mentions a crop model, DSSAT, irrigation or drought, those three are not optional,
   because field capacity minus wilting point is what makes the soil hold water for a plant.

4. VALIDATE, THEN INTERPRET. `download_soil` returns a verdict. If it FAILED, relay the problems
   and stop - do not plot a cube that did not pass. If it PASSED, call `available_water` and
   `plot_soil` once each, then describe what the numbers say: the texture class the fractions
   imply, whether pH is limiting, how much water the profile holds, and how the properties change
   with depth. You cannot see the images, so work from the returned numbers.

5. YOU HAVE NO CHAT CHANNEL. The only way to ask anything is `ask_user`, which takes a LIST of
   questions. A question written as message text will never be answered, so never end a message
   with one. Batch everything: the user pays per turn. Your first ask_user call should carry
   every parameter you still need. Aim to finish in under six turns.

PARAMETERS

  Propose, then confirm: the variable list, the depths, and a suffix (a short slug like
  yoro_hnd). Never guess the area.

  SoilGrids variables: clay, sand, silt, bdod, cfvo, soc, phh2o (general); wv0010, wv0033,
  wv1500 (water retention); nitrogen, cec (on request). Depths: 0-5, 5-15, 15-30, 30-60,
  60-100 cm. Anything else is rejected by the tool, so do not invent names. ocd and ocs exist
  in the SoilGrids catalogue but are not wired into aggeodata - if asked, say so plainly;
  requesting them would produce a cube quietly missing those layers.

  THE TWO GROUPS ARE AT DIFFERENT RESOLUTIONS. The general properties come from the 250 m
  coverage API; wv0010/wv0033/wv1500 are published ONLY at 1 km - there is no 250 m version to
  ask for. A cube has one grid, so reference_variable decides which group gets resampled. The
  default wv1500 downsamples the 250 m layers to 1 km: nothing invented, detail averaged away.
  Naming a 250 m variable instead upsamples the water layers, which looks sharper than the data
  really is. For crop modeling keep wv1500 - the cube gets resampled onto a coarser climate grid
  anyway. For soil mapping, where texture detail is the point, prefer a 250 m reference. When a
  request mixes the groups, say which way it went and why; download_soil returns that note.

  Units are stored as scaled integers - clay in g/kg, pH times 10, water contents times 1000.
  The notebook converts them for display; quote readable units to the user (38% clay, pH 6.8,
  0.21 cm3/cm3), never the raw integers.

AFTER THE RUN

  Say where the cube is, what shape it has, and what it is made of. Close with the honest limits:
  SoilGrids is a global model, not a soil survey, so it is a prior for a place rather than a
  measurement of a field, and even 250 m smooths over the variation a farmer would see across
  one plot. If the cube includes water-retention layers, say that those came from the 1 km
  product and were resampled - they are coarser than the texture fractions beside them.

--- FULL SKILL BELOW ---
"""

if SKILL_TEXT:
    SYSTEM_INSTRUCTION = BRIEF + "\n" + SKILL_TEXT
    print(f"System instruction: {len(SYSTEM_INSTRUCTION):,} chars "
          f"({len(BRIEF):,} brief + {len(SKILL_TEXT):,} skill)")

In [ ]:
#@title Create the agent { display-mode: "form" }
if not SYSTEM_INSTRUCTION:
    raise RuntimeError("No system instruction - the skill fetch in the cell above did not "
                       "succeed. Re-run that cell before creating the agent.")

soil_agent = client.agents.create(
    id=f"soil-data-download-{uuid.uuid4().hex[:8]}",
    base_agent=AGENT,
    system_instruction=SYSTEM_INSTRUCTION,
    base_environment={"type": "remote"},
)
print(f"Agent ready: {soil_agent.id}")

### Your request &nbsp; ✏️ **EDIT ME**

A place is enough — the agent proposes the crop-modeling variable set and all five depths, and
asks before downloading.

| You want | Write something like |
|---|---|
| Everything a crop model needs | `the soil a crop model needs for Yoro, Honduras` |
| One district | `SoilGrids for Dedza district, Malawi` |
| A subset | `just clay, sand and silt for bbox [-87.8, 14.0, -86.8, 15.0], top 30 cm` |
| Carbon work | `organic carbon and pH for Rwanda, all depths` |


In [ ]:
PROMPT = "The soil a crop model needs for Yoro, Honduras" #@param {type:"string"}

start(PROMPT)

---
## 5 · The maps again

The agent draws these during the conversation. Run this to redraw or export them without spending
another API turn.


In [ ]:
#@title Redraw and list the outputs { display-mode: "form" }
if state.get("cube"):
    summary = plot_soil()
    for var, s in summary.get("variables", {}).items():
        if var.startswith("_"):
            continue
        print(f"  {var:9} ({s['unit']:>9})  " +
              "  ".join(f"{d}:{v}" for d, v in s["by_depth"].items()))
    awc = summary.get("available_water", {})
    if awc.get("profile_total_mm") is not None:
        print(f"\n  plant-available water, 0-100 cm: {awc['profile_total_mm']} mm "
              f"(range {awc['min_mm']}-{awc['max_mm']})")
    print(f"\ncube: {state['cube']}")
    for p in sorted(pathlib.Path(state["cube"]).parent.glob("*.png")):
        print(f"  figure: {p.name}")
else:
    print("No download in this session yet - run the cells above first.")

---
## What next?

- **Pair it with weather.** The
  [`climate-data-download` notebook](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/skills/blob/main/notebooks/climate_data_download_example.ipynb)
  builds the matching climate cube. For a crop model both cubes must sit on the **same grid** —
  SoilGrids is 250 m (1 km for the water layers) and CHIRPS is 5 km, so the soil cube is
  resampled onto the climate grid before simulating, not the other way round.
- **Run a crop model** with the
  [`spatial-crop-modeler` notebook](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/skills/blob/main/notebooks/spatial_crop_modeler_example.ipynb),
  which takes a weather cube and a soil cube and runs DSSAT over them.
- **Document it** with the
  [`cdh-metadata` notebook](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/skills/blob/main/notebooks/CDH_metadata_example.ipynb).
- **Save it before the runtime recycles.** Colab discards `/content` when the session ends.
- **Read the skill**: [soil-data-download](https://github.com/CGIAR-Climate-Data-Hub/skills/blob/main/.agents/skills/soil-data-download/SKILL.md)
  · [SoilGrids](https://soilgrids.org/) · [aggeodata](https://github.com/CGIAR-Climate-Data-Hub/aggeodata)
- **More skills**: [CGIAR-Climate-Data-Hub/skills](https://github.com/CGIAR-Climate-Data-Hub/skills)
